In [10]:
from dotenv import load_dotenv
import os

load_dotenv()
print("Env loaded:", bool(os.getenv("GROQ_API_KEY")))

Env loaded: True


In [11]:
from github import Auth, Github

auth = Auth.Token(os.getenv("GITHUB_PAT"))
gh = Github(auth=auth)

repo = gh.get_repo("PrashantAghara/fastapi")
pulls = repo.get_pulls(state="open")
pr = pulls[0]
files = pr.get_files()

for f in files:
    print(f.filename, f.status, f.additions, f.deletions)

fastapi/applications.py modified 1 0


In [12]:
import re, subprocess, json, tempfile, os


def get_changed_line_ranges(patch: str) -> list[tuple[int, int]]:
    """Extract (start, end) line ranges of the new file version from a unified diff hunk header."""
    ranges = []
    for match in re.finditer(r"@@ -\d+,?\d* \+(\d+),?(\d*) @@", patch):
        start = int(match.group(1))
        length = int(match.group(2)) if match.group(2) else 1
        ranges.append((start, start + length - 1))
    return ranges


def run_ruff_on_file(filename: str, full_content: str) -> list[dict]:
    with tempfile.NamedTemporaryFile(
        suffix=".py", delete=False, mode="w", encoding="utf-8"
    ) as tmp:
        tmp.write(full_content)
        tmp_path = tmp.name
    result = subprocess.run(
        ["ruff", "check", tmp_path, "--output-format=json"],
        capture_output=True,
        text=True,
    )
    os.unlink(tmp_path)
    try:
        return json.loads(result.stdout) if result.stdout else []
    except json.JSONDecodeError:
        return []


def filter_to_diff(findings: list[dict], ranges: list[tuple[int, int]]) -> list[dict]:
    return [
        f for f in findings if any(s <= f["location"]["row"] <= e for s, e in ranges)
    ]

In [13]:
from langchain_core.tools import tool


@tool
def static_analysis_tool(filename: str) -> dict:
    """Run ruff against this PR's version of a file, scoped to only the changed lines."""
    file_obj = next(f for f in pr.get_files() if f.filename == filename)
    full_content = repo.get_contents(filename, ref=pr.head.sha).decoded_content.decode(
        "utf-8"
    )
    raw = run_ruff_on_file(filename, full_content)
    ranges = get_changed_line_ranges(file_obj.patch)
    return {"filename": filename, "findings": filter_to_diff(raw, ranges)}

In [14]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

py_filenames = [f.filename for f in pr.get_files() if f.filename.endswith(".py")]

static_analysis_agent = create_agent(
    model=llm,
    tools=[static_analysis_tool],
    system_prompt=(
        "You are a Static Analysis Agent reviewing a pull request. "
        "You are given a list of changed Python filenames. "
        "Call static_analysis_tool once per filename to get lint findings scoped to the changed lines. "
        "When summarizing, use the EXACT 'location.row' value from each finding as the Line number — "
        "do not estimate, renumber, or invent line numbers. "
        "Summarize findings grouped by severity, and end with a one-line verdict: "
        "PASS, PASS_WITH_WARNINGS, or FAIL."
    ),
)

result = static_analysis_agent.invoke(
    {"messages": [{"role": "user", "content": f"Changed Python files: {py_filenames}"}]}
)

print(result["messages"][-1].content)

**Findings for `fastapi/applications.py`**

**Error (severity: error)**
- Line **4775**: `uuid` imported but unused (`F401`, unused-import)

**Verdict:** FAIL


In [17]:
STYLE_GUIDE_PLACEHOLDER = """
- Function and variable names should be descriptive, not abbreviated (e.g. `user_id` not `uid`)
- Public functions should have docstrings describing purpose, args, and return value
- Avoid deeply nested conditionals (max 3 levels) — prefer early returns
- Type hints are required on all function signatures
- No commented-out code should be left in
"""  # Placeholder — real ingestion (repo's actual CONTRIBUTING.md/style docs) comes in Phase 2 RAG

def get_pr_diff_text(pr, filenames: list[str]) -> str:
    diff_parts = []
    for f in pr.get_files():
        if f.filename in filenames:
            diff_parts.append(f"--- {f.filename} ---\n{f.patch}")
    return "\n\n".join(diff_parts)

style_prompt = """You are a Style Agent reviewing a pull request diff against a style guide.

Style guide:
{style_guide}

Diff:
{diff}

For each style issue found, output: filename, line (from the diff hunk header context — reference the nearest @@ line marker), a short comment, and severity (info/warning).
If no issues, say so explicitly. End with a one-line verdict: PASS, PASS_WITH_WARNINGS, or FAIL.
Do not comment on anything the style guide doesn't cover."""

diff_text = get_pr_diff_text(pr, py_filenames)

style_result = llm.invoke(
    style_prompt.format(style_guide=STYLE_GUIDE_PLACEHOLDER, diff=diff_text)
)
print(style_result.content)

No style issues found.

PASS


In [18]:
pulls = repo.get_pulls(state="open")
style_pr = next(p for p in pulls if p.head.ref == "test/style-violation")
py_filenames = [f.filename for f in style_pr.get_files() if f.filename.endswith(".py")]
diff_text = get_pr_diff_text(style_pr, py_filenames)

style_result = llm.invoke(
    style_prompt.format(style_guide=STYLE_GUIDE_PLACEHOLDER, diff=diff_text)
)
print(style_result.content)

fastapi/applications.py, line 4772: Function name `proc` is not descriptive. warning  
fastapi/applications.py, line 4772: Parameter names `d` and `u` are not descriptive. warning  
fastapi/applications.py, line 4772: Missing type hints on function signature. warning  
fastapi/applications.py, line 4772: Missing docstring for public function. warning  
fastapi/applications.py, line 4772: Nested conditionals could be simplified with early returns. warning  

PASS_WITH_WARNINGS
